# Junção das bases Scopus e Web of Science

Este notebook carrega os arquivos exportados das pastas `scopus` e `wos`, monta um resumo padronizado dos artigos, apresenta os quantitativos de carregamento, une as bases e separa os registros duplicados dos não duplicados.

> Observação: o preenchimento de `journal_impact_factor` acontece apenas na base sem duplicados e somente quando a variável `CONSULTAR_FATOR_IMPACTO` estiver ativada. Por padrão, a consulta fica desligada.

In [4]:
from pathlib import Path
import json
import re
import unicodedata
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', None)

In [6]:
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
SCOPUS_DIR = BASE_DIR / 'dados' / 'base_dados' / 'scopus'
WOS_DIR = BASE_DIR / 'dados' / 'base_dados' / 'wos'
OUTPUT_DIR = (BASE_DIR / 'dados' / 'identificados').resolve()

CONSULTAR_FATOR_IMPACTO = False
OPENALEX_EMAIL = ''  # opcional: informe seu e-mail para pool polite da API

scopus_files = sorted(SCOPUS_DIR.glob('*.csv'))
wos_files = sorted(WOS_DIR.glob('*.txt'))

if not scopus_files:
    raise FileNotFoundError(f'Nenhum arquivo .csv encontrado em {SCOPUS_DIR}')
if not wos_files:
    raise FileNotFoundError(f'Nenhum arquivo .txt encontrado em {WOS_DIR}')

pd.DataFrame(
    {
        'base': ['Scopus', 'Web of Science'],
        'quantidade_arquivos': [len(scopus_files), len(wos_files)],
        'arquivos': [
            ', '.join(path.name for path in scopus_files),
            ', '.join(path.name for path in wos_files),
        ],
    }
)

,base,quantidade_arquivos,arquivos
0,Scopus,4,"AG_Scopus.csv, CA_Scopus.csv, CAS_Scopus.csv, CS_Scopus.csv"
1,Web of Science,2,"CA_WOS.txt, CS_WOS.txt"


In [7]:
def parse_wos_txt(path: Path) -> list[dict]:
    records = []
    current_record = {}
    current_tag = None

    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.rstrip('\n')

        if not line.strip():
            continue

        if line.startswith('ER'):
            if current_record:
                records.append(current_record)
                current_record = {}
            current_tag = None
            continue

        if re.match(r'^[A-Z0-9]{2} ', line):
            tag = line[:2]
            value = line[3:].strip()
            current_record.setdefault(tag, []).append(value)
            current_tag = tag
        elif line.startswith('   ') and current_tag:
            current_record[current_tag].append(line.strip())

    return records


def join_values(record: dict, tag: str) -> str:
    return ' '.join(record.get(tag, [])).strip()


def first_value(record: dict, tag: str) -> str:
    values = record.get(tag, [])
    return values[0].strip() if values else ''


def clean_abstract(text: str) -> str:
    text = '' if pd.isna(text) else str(text).strip()
    text = re.sub(r'^(abstract\s*[:.-]?\s*)', '', text, flags=re.IGNORECASE)
    return re.sub(r'\s+', ' ', text).strip()


def normalize_text(value) -> str:
    if pd.isna(value) or value is None:
        return ''
    value = str(value).strip().lower()
    value = unicodedata.normalize('NFKD', value)
    value = ''.join(char for char in value if not unicodedata.combining(char))
    value = re.sub(r'\W+', ' ', value)
    return re.sub(r'\s+', ' ', value).strip()


def extract_intersection_name(source_file: str) -> str:
    if pd.isna(source_file) or source_file is None:
        return ''

    file_name = Path(str(source_file).strip()).name
    stem = Path(file_name).stem
    match = re.match(r'^([^_]+)', stem)
    return match.group(1) if match else stem


def build_openalex_url(journal_name: str, email: str = '') -> str:
    params = {
        'search': journal_name,
        'filter': 'type:journal',
        'per-page': 5,
    }
    if email:
        params['mailto'] = email
    return f"https://api.openalex.org/sources?{urlencode(params)}"


def extract_openalex_impact_factor(result: dict):
    summary_stats = result.get('summary_stats') or {}
    value = summary_stats.get('2yr_mean_citedness')
    return round(value, 3) if value is not None else pd.NA


def fetch_impact_factor_from_openalex(journal_name: str, email: str = ''):
    normalized_target = normalize_text(journal_name)
    if not normalized_target:
        return pd.NA

    request = Request(
        build_openalex_url(journal_name, email=email),
        headers={'User-Agent': 'revisor-notebook/1.0'},
    )

    try:
        with urlopen(request, timeout=20) as response:
            payload = json.loads(response.read().decode('utf-8'))
    except (HTTPError, URLError, TimeoutError, json.JSONDecodeError):
        return pd.NA

    results = payload.get('results', [])
    if not results:
        return pd.NA

    for result in results:
        display_name = normalize_text(result.get('display_name', ''))
        if display_name == normalized_target:
            return extract_openalex_impact_factor(result)

    return pd.NA


def build_impact_factor_map(journal_names, email: str = '') -> dict:
    unique_journals = sorted(
        {
            str(name).strip()
            for name in journal_names
            if pd.notna(name) and str(name).strip()
        }
    )

    impact_factor_map = {}
    total = len(unique_journals)

    for index, journal_name in enumerate(unique_journals, start=1):
        impact_factor_map[journal_name] = fetch_impact_factor_from_openalex(journal_name, email=email)
        if index == 1 or index % 25 == 0 or index == total:
            print(f'Consulta de fator de impacto: {index}/{total}')

    return impact_factor_map


def load_scopus_files(file_paths) -> pd.DataFrame:
    frames = []
    for file_path in file_paths:
        frame = pd.read_csv(file_path)
        frame['source_file'] = file_path.name
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


def load_wos_files(file_paths) -> list[dict]:
    records = []
    for file_path in file_paths:
        file_records = parse_wos_txt(file_path)
        for record in file_records:
            record['source_file'] = [file_path.name]
        records.extend(file_records)
    return records

In [8]:
scopus_raw = load_scopus_files(scopus_files)
wos_records = load_wos_files(wos_files)

scopus_resumo = pd.DataFrame(
    {
        'title': scopus_raw['Title'],
        'first_author': scopus_raw['Authors'].fillna('').astype(str).str.split(';').str[0].str.strip(),
        'publication_year': pd.to_numeric(scopus_raw['Year'], errors='coerce').astype('Int64'),
        'journal': scopus_raw['Source title'],
        'journal_impact_factor': pd.Series([pd.NA] * len(scopus_raw), dtype='Float64'),
        'abstract': scopus_raw['Abstract'].map(clean_abstract),
        'doi': scopus_raw['DOI'],
        'denominacao_intersecao': scopus_raw['source_file'].map(extract_intersection_name),
        'source_db': 'Scopus',
        'source_file': scopus_raw['source_file'],
    }
)

wos_resumo = pd.DataFrame(
    {
        'title': [join_values(record, 'TI') for record in wos_records],
        'first_author': [first_value(record, 'AU') for record in wos_records],
        'publication_year': pd.Series([first_value(record, 'PY') for record in wos_records], dtype='Int64'),
        'journal': [join_values(record, 'SO') for record in wos_records],
        'journal_impact_factor': pd.Series([pd.NA] * len(wos_records), dtype='Float64'),
        'abstract': [clean_abstract(join_values(record, 'AB')) for record in wos_records],
        'doi': [first_value(record, 'DI') for record in wos_records],
        'denominacao_intersecao': [extract_intersection_name(first_value(record, 'source_file')) for record in wos_records],
        'source_db': 'Web of Science',
        'source_file': [first_value(record, 'source_file') for record in wos_records],
    }
)

colunas_resumo = [
    'title',
    'first_author',
    'publication_year',
    'journal',
    'journal_impact_factor',
    'abstract',
    'doi',
    'denominacao_intersecao',
    'source_db',
    'source_file',
]

scopus_resumo = scopus_resumo[colunas_resumo]
wos_resumo = wos_resumo[colunas_resumo]

In [9]:
quantitativo_carregamento = pd.DataFrame(
    {
        'base': ['Scopus', 'Web of Science'],
        'quantidade_artigos': [len(scopus_resumo), len(wos_resumo)],
    }
)

display(quantitativo_carregamento)

,base,quantidade_artigos
0,Scopus,7474
1,Web of Science,86


In [12]:
scopus_resumo['denominacao_intersecao'].value_counts()

denominacao_intersecao
CS     4654
CA     2630
AG      100
CAS      90
Name: count, dtype: int64

In [13]:
wos_resumo['denominacao_intersecao'].value_counts()

denominacao_intersecao
CS    50
CA    36
Name: count, dtype: int64

In [37]:
base_unificada = pd.concat([scopus_resumo, wos_resumo], ignore_index=True)

base_unificada['doi_key'] = base_unificada['doi'].fillna('').astype(str).str.strip().str.lower()
base_unificada['fallback_key'] = base_unificada.apply(
    lambda row: ' | '.join(
        [
            normalize_text(row['title']),
            normalize_text(row['first_author']),
            normalize_text(row['publication_year']),
        ]
    ),
    axis=1,
)
base_unificada['match_key'] = base_unificada['doi_key'].where(
    base_unificada['doi_key'] != '',
    base_unificada['fallback_key'],
)

base_unificada['prioridade_deduplicacao'] = base_unificada['denominacao_intersecao'].ne('CAS').astype(int)
base_unificada = base_unificada.sort_values(
    ['match_key', 'prioridade_deduplicacao', 'source_db', 'title'],
    kind='stable',
).reset_index(drop=True)

duplicados = base_unificada.loc[
    base_unificada.duplicated('match_key', keep=False)
].sort_values(['match_key', 'prioridade_deduplicacao', 'source_db', 'title'], kind='stable').reset_index(drop=True)

base_sem_duplicados = base_unificada.drop_duplicates('match_key', keep='first').reset_index(drop=True)
base_unificada = base_unificada.drop(columns=['prioridade_deduplicacao'])
duplicados = duplicados.drop(columns=['prioridade_deduplicacao'])
base_sem_duplicados = base_sem_duplicados.drop(columns=['prioridade_deduplicacao'])

if CONSULTAR_FATOR_IMPACTO:
    impact_factor_map = build_impact_factor_map(base_sem_duplicados['journal'], email=OPENALEX_EMAIL)
    base_sem_duplicados['journal_impact_factor'] = base_sem_duplicados['journal'].map(impact_factor_map).astype('Float64')
else:
    base_sem_duplicados['journal_impact_factor'] = pd.Series([pd.NA] * len(base_sem_duplicados), dtype='Float64')

resumo_uniao = pd.DataFrame(
    {
        'indicador': [
            'quantidade_total_apos_juncao',
            'quantidade_de_linhas_em_grupos_duplicados',
            'quantidade_de_grupos_duplicados',
            'quantidade_excluida_na_deduplicacao',
            'quantidade_mantida_apos_deduplicacao',
            'quantidade_com_fator_impacto_na_base_final',
        ],
        'valor': [
            len(base_unificada),
            len(duplicados),
            duplicados['match_key'].nunique(),
            len(base_unificada) - len(base_sem_duplicados),
            len(base_sem_duplicados),
            base_sem_duplicados['journal_impact_factor'].notna().sum(),
        ],
    }
)

display(resumo_uniao)

Consulta de fator de impacto: 1/1927
Consulta de fator de impacto: 25/1927
Consulta de fator de impacto: 50/1927
Consulta de fator de impacto: 75/1927
Consulta de fator de impacto: 100/1927
Consulta de fator de impacto: 125/1927
Consulta de fator de impacto: 150/1927
Consulta de fator de impacto: 175/1927
Consulta de fator de impacto: 200/1927
Consulta de fator de impacto: 225/1927
Consulta de fator de impacto: 250/1927
Consulta de fator de impacto: 275/1927
Consulta de fator de impacto: 300/1927
Consulta de fator de impacto: 325/1927
Consulta de fator de impacto: 350/1927
Consulta de fator de impacto: 375/1927
Consulta de fator de impacto: 400/1927
Consulta de fator de impacto: 425/1927
Consulta de fator de impacto: 450/1927
Consulta de fator de impacto: 475/1927
Consulta de fator de impacto: 500/1927
Consulta de fator de impacto: 525/1927
Consulta de fator de impacto: 550/1927
Consulta de fator de impacto: 575/1927
Consulta de fator de impacto: 600/1927
Consulta de fator de impacto: 

,indicador,valor
0,quantidade_total_apos_juncao,7560
1,quantidade_de_linhas_em_grupos_duplicados,434
2,quantidade_de_grupos_duplicados,126
3,quantidade_excluida_na_deduplicacao,308
4,quantidade_mantida_apos_deduplicacao,7252
5,quantidade_com_fator_impacto_na_base_final,1759


In [38]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

base_unificada_path = OUTPUT_DIR / 'base_unificada.csv'
base_duplicados_path = OUTPUT_DIR / 'base_duplicados.csv'
base_sem_duplicados_path = OUTPUT_DIR / 'base_sem_duplicados.csv'

base_unificada[colunas_resumo].to_csv(base_unificada_path, index=False, encoding='utf-8-sig')
duplicados[colunas_resumo + ['match_key']].to_csv(base_duplicados_path, index=False, encoding='utf-8-sig')
base_sem_duplicados[colunas_resumo].to_csv(base_sem_duplicados_path, index=False, encoding='utf-8-sig')

arquivos_salvos = pd.DataFrame(
    {
        'dataset': ['base_unificada', 'base_duplicados', 'base_sem_duplicados'],
        'arquivo': [
            str(base_unificada_path),
            str(base_duplicados_path),
            str(base_sem_duplicados_path),
        ],
    }
)

display(arquivos_salvos)

,dataset,arquivo
0,base_unificada,C:\Users\ultra\OneDrive\Documentos\github\revisor\dados\identificados\base_unificada.csv
1,base_duplicados,C:\Users\ultra\OneDrive\Documentos\github\revisor\dados\identificados\base_duplicados.csv
2,base_sem_duplicados,C:\Users\ultra\OneDrive\Documentos\github\revisor\dados\identificados\base_sem_duplicados.csv


In [39]:
display(base_unificada[colunas_resumo].head(2))

,title,first_author,publication_year,journal,journal_impact_factor,abstract,doi,denominacao_intersecao,source_db,source_file
0,03-005-Indicators of perceived quality in urban services: qualitative analysisapplied to street cleaning and waste c...,Heras-Romanos E.,2025,Proceedings from the International Congress on Project Management and Engineering,<NA>,"The management of street cleaning is an essential service in urban environments, and its rigorous evaluation is key ...",NaN,CS,Scopus,CS_Scopus.csv
1,"""I can't get no satisfaction:"" The impact of personality and emotion on postpurchase processes",Mooradian T.A.,1997,Psychology and Marketing,<NA>,Emerging theory and empirics in personality psychology have related enduring traits with transient affective experie...,10.1002/(SICI)1520-6793(199707)14:4<379::AID-MAR5>3.0.CO;2-6,CS,Scopus,CS_Scopus.csv


In [40]:
display(duplicados[colunas_resumo + ['match_key']].head(2))

,title,first_author,publication_year,journal,journal_impact_factor,abstract,doi,denominacao_intersecao,source_db,source_file,match_key
0,"Exploring the relationship between chatbots, service failure recovery and customer loyalty: A frustration–aggression...",Ozuem W.,2024,Psychology and Marketing,<NA>,An increasing number of companies are introducing chatbot-led contexts in service failure recovery. Existing studies...,10.1002/mar.22051,CAS,Scopus,CAS_Scopus.csv,10.1002/mar.22051
1,"Exploring the relationship between chatbots, service failure recovery and customer loyalty: A frustration–aggression...",Ozuem W.,2024,Psychology and Marketing,<NA>,An increasing number of companies are introducing chatbot-led contexts in service failure recovery. Existing studies...,10.1002/mar.22051,AG,Scopus,AG_Scopus.csv,10.1002/mar.22051


In [41]:
display(base_sem_duplicados[colunas_resumo].head(2))

,title,first_author,publication_year,journal,journal_impact_factor,abstract,doi,denominacao_intersecao,source_db,source_file
0,03-005-Indicators of perceived quality in urban services: qualitative analysisapplied to street cleaning and waste c...,Heras-Romanos E.,2025,Proceedings from the International Congress on Project Management and Engineering,<NA>,"The management of street cleaning is an essential service in urban environments, and its rigorous evaluation is key ...",NaN,CS,Scopus,CS_Scopus.csv
1,"""I can't get no satisfaction:"" The impact of personality and emotion on postpurchase processes",Mooradian T.A.,1997,Psychology and Marketing,<NA>,Emerging theory and empirics in personality psychology have related enduring traits with transient affective experie...,10.1002/(SICI)1520-6793(199707)14:4<379::AID-MAR5>3.0.CO;2-6,CS,Scopus,CS_Scopus.csv


In [42]:
base_sem_duplicados['denominacao_intersecao'].unique()

array(['CS', 'CA', 'CAS', 'AG'], dtype=object)

In [43]:
base_unificada['denominacao_intersecao'].unique()

array(['CS', 'CA', 'CAS', 'AG'], dtype=object)

In [44]:
df = pd.read_csv('../dados/identificados/base_sem_duplicados.csv')
df['journal_impact_factor'].unique()

array([   nan,  1.769,  0.835,  1.627,  5.957,  3.874,  4.554,  2.36 ,
        1.86 ,  0.   ,  2.055,  5.714,  1.909,  2.536,  1.932,  6.305,
        5.482,  4.84 ,  6.29 ,  4.463, 12.838,  6.448,  9.   ,  4.49 ,
        3.63 ,  8.809,  3.217,  5.624,  4.966,  6.551,  5.267,  2.911,
        0.422,  5.315,  6.16 ,  1.713,  7.863,  5.791,  5.382,  2.049,
        3.978, 11.903,  2.182,  1.276,  6.515,  2.098,  5.77 ,  4.592,
        1.068,  5.736,  2.447,  6.41 ,  2.88 ,  5.186,  1.886,  2.7  ,
        1.269,  3.288,  4.705,  1.585,  3.607,  2.228,  4.101,  4.114,
        1.603,  2.179,  1.95 ,  3.265,  2.05 ,  0.403,  1.155,  1.333,
        3.   ,  4.   ,  5.004,  3.764,  3.806,  2.843,  3.895,  0.459,
        3.625,  3.775,  9.461,  5.59 ,  0.033,  1.434,  3.858,  2.469,
        0.077,  5.171,  1.444,  2.313,  4.432,  3.27 ,  3.119,  1.703,
        0.245,  2.748,  5.175,  2.504,  2.588,  3.954,  7.105,  3.418,
        1.64 ,  4.439,  3.03 ,  5.151,  1.546,  1.571,  4.214,  3.466,
      

In [46]:
base_sem_duplicados['journal_impact_factor'].unique()

<FloatingArray>
[  <NA>,  1.769,  0.835,  1.627,  5.957,  3.874,  4.554,   2.36,   1.86,
    0.0,
 ...
  0.582,  1.194,  0.213,  0.532,  0.127, 26.323,   0.09,  1.306,  0.575,
  0.444]
Length: 300, dtype: Float64